This document explores an assignment: https://huggingface.co/learn/audio-course/chapter5/hands_on 

Here are the instructions:

- Fine-tune the ”openai/whisper-tiny” model using the American English (“en-US”) subset of the ”PolyAI/minds14” dataset.
- Use the first 450 examples for training, and the rest for evaluation. Ensure you set num_proc=1 when pre-processing the dataset using the .map method (this will ensure your model is submitted correctly for assessment).

To evaluate the model, use the wer and wer_ortho metrics as described in this Unit. However, do not convert the metric into percentages by multiplying by 100 (E.g. if WER is 42%, we’ll expect to see the value of 0.42 in this exercise).
Once you have fine-tuned a model, make sure to upload it to the 🤗 Hub with the following kwargs:

Copied
kwargs = {
     "dataset_tags": "PolyAI/minds14",
    "finetuned_from": "openai/whisper-tiny",
    "tasks": "automatic-speech-recognition",
}

In [21]:
from dataclasses import dataclass
from functools import partial
from typing import Any, Dict, List, Union

import evaluate
import torch
from datasets import Audio, load_dataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    set_seed,
)
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

set_seed(42)
MODEL_ID = "openai/whisper-tiny"
LANGUAGE = "english"
TASK = "transcribe"

In [22]:
dataset = load_dataset("PolyAI/minds14", "en-US", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

train_dataset = dataset.select(range(450))
eval_dataset = dataset.select(range(450, len(dataset)))

assert len(train_dataset) == 450
assert len(eval_dataset) == 113

In [23]:
assert len(train_dataset) == 450
assert len(eval_dataset) == 113

Load the processor below

In [24]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, Language= LANGUAGE, task=TASK)
print(f"Processor: {processor.__class__.__name__}")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
print(model.config)

Processor: WhisperProcessor


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 11363.17it/s]


WhisperConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "apply_spec_augment": false,
  "architectures": [
    "WhisperForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "begin_suppress_tokens": [
    220,
    50257
  ],
  "bos_token_id": 50257,
  "classifier_proj_size": 256,
  "d_model": 384,
  "decoder_attention_heads": 6,
  "decoder_ffn_dim": 1536,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 4,
  "decoder_start_token_id": 50258,
  "dropout": 0.0,
  "dtype": "float32",
  "encoder_attention_heads": 6,
  "encoder_ffn_dim": 1536,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 4,
  "eos_token_id": 50257,
  "forced_decoder_ids": [
    [
      1,
      50259
    ],
    [
      2,
      50359
    ],
    [
      3,
      50363
    ]
  ],
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "mask_feature_length": 10,
  "mask_feature_min_masks": 0,
  "mask_feature_prob": 0.0,
  "mask_time_length": 10,
  "mask_time_min_masks": 2,
  "mask_time_prob": 0.05,
  

In [25]:
train_dataset.column_names
 

['path',
 'audio',
 'transcription',
 'english_transcription',
 'intent_class',
 'lang_id']

In [26]:
from IPython.display import Audio, display
 

In [27]:
train_dataset


Dataset({
    features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
    num_rows: 450
})

In [28]:
print(train_dataset)
print(train_dataset.column_names)
print(train_dataset.features)
print(train_dataset.num_rows)
print(train_dataset.shape)

Dataset({
    features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
    num_rows: 450
})
['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id']
{'path': Value('string'), 'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'transcription': Value('string'), 'english_transcription': Value('string'), 'intent_class': ClassLabel(names=['abroad', 'address', 'app_error', 'atm_limit', 'balance', 'business_loan', 'card_issues', 'cash_deposit', 'direct_debit', 'freeze', 'high_value_payment', 'joint_account', 'latest_transactions', 'pay_bill']), 'lang_id': ClassLabel(names=['cs-CZ', 'de-DE', 'en-AU', 'en-GB', 'en-US', 'es-ES', 'fr-FR', 'it-IT', 'ko-KR', 'nl-NL', 'pl-PL', 'pt-PT', 'ru-RU', 'zh-CN'])}
450
(450, 6)


In [29]:
train_dataset[0]

{'path': 'en-US~JOINT_ACCOUNT/602ba55abb1e6d0fbce92065.wav',
 'audio': <datasets.features._torchcodec.AudioDecoder at 0x1aad9d83820>,
 'transcription': 'I would like to set up a joint account with my partner',
 'english_transcription': 'I would like to set up a joint account with my partner',
 'intent_class': 11,
 'lang_id': 4}

In [30]:
display(Audio(train_dataset[0]["audio"]["array"], rate=train_dataset[0]["audio"]["sampling_rate"]))

In [31]:
sample = train_dataset[0]
decoder = sample["audio"]

audio_samples = decoder.get_all_samples()
waveform = audio_samples.data
sample_rate = audio_samples.sample_rate

In [32]:
print("processor" in globals())
print(processor)

True
WhisperProcessor:
- feature_extractor: WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}

- tokenizer: WhisperTokenizer(name_or_path='openai/whisper-tiny', vocab_size=50258, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50257: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50258: AddedToken("<|startoftranscript|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50259: AddedToken("<|en|>", rstrip=False, lstrip=False, singl

In [33]:
def prepare_example(batch, processor: WhisperProcessor):
    audio = batch["audio"] 

    # compute log-Mel input features from input audio array
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features

    # encode target text to label ids
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

train_dataset = train_dataset.map(prepare_example,fn_kwargs={"processor": processor}, num_proc=1, remove_columns = train_dataset.column_names, desc="preprocess train dataset")

eval_dataset = eval_dataset.map(prepare_example,fn_kwargs={"processor": processor}, num_proc=1, remove_columns = eval_dataset.column_names, desc="preprocess eval dataset")

In [34]:
train_dataset 

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 450
})

In [35]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_features": feature["input_features"][0]} for feature in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [36]:
wer_metric = evaluate.load("wer")
normalizer = BasicTextNormalizer()

In [37]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    predictions = processor.batch_decode(pred_ids, skip_special_tokens=True)
    references = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer_ortho = wer_metric.compute(predictions=predictions, references=references)

    normalized_pairs = [
        (normalizer(prediction), normalizer(reference))
        for prediction, reference in zip(predictions, references)
        if normalizer(reference).strip()
    ]
    normalized_predictions, normalized_references = map(list, zip(*normalized_pairs))
    wer = wer_metric.compute(
        predictions=normalized_predictions, references=normalized_references
    )
    return {"wer": wer, "wer_ortho": wer_ortho}

In [38]:
model.config.use_cache = False
model.generate = partial(
    model.generate,
    language=LANGUAGE,
    task=TASK,
    use_cache=True,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="outputs/whisper-tiny-minds14-en-us",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=50,
    max_steps=500,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps", 
    save_steps=100,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=False,
    hub_model_id="whisper-tiny-minds14-en-us",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [19]:
import torch 

torch.cuda.is_available()

True

In [ ]:
trainer.train()

metrics = trainer.evaluate()
print(metrics)  # inspect eval_wer and eval_wer_ortho as fractions


Step,Training Loss,Validation Loss,Wer,Wer Ortho
100,0.282948,0.515898,0.318182,0.358421
200,0.033673,0.581166,0.340614,0.375694
300,0.004466,0.638816,0.333530,0.362739
400,0.001089,0.664701,0.332349,0.354719


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform